**Steps**

1. Load text, image and structured features.
2. Align modalities using study_id.
3. Combine the feature representations.
4. Apply a simple class-balancing step.
5. Train the multimodal fusion classifier.
6. Evaluate the Layer 1 model.
7. Save the trained model and fused features.

In [1]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

from google.colab import drive

drive.mount(
    '/content/drive'
)

Mounted at /content/drive


In [2]:
base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

os.makedirs(
    model_path,
    exist_ok=True
)

print(
    "Processed path:",
    processed_path
)

print(
    "Model path:",
    model_path
)

Processed path: /content/drive/MyDrive/dissertation_project/data/processed
Model path: /content/drive/MyDrive/dissertation_project/data/models


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print(
    "Device:",
    device
)

Device: cuda


In [4]:
# Load text features
text_dev_file = (
    f'{processed_path}/'
    'text_features_development.csv'
)

text_heldout_file = (
    f'{processed_path}/'
    'text_features_heldout.csv'
)

text_dev = pd.read_csv(
    text_dev_file
)

text_heldout = pd.read_csv(
    text_heldout_file
)

print(
    "Text development:",
    text_dev.shape
)

print(
    "Text held-out:",
    text_heldout.shape
)

print(
    "\nText columns:",
    len(text_dev.columns)
)

Text development: (1513, 769)
Text held-out: (687, 769)

Text columns: 769


In [5]:
# Load image features
image_dev_file = (
    f'{processed_path}/'
    'image_features_development.csv'
)

image_heldout_file = (
    f'{processed_path}/'
    'image_features_heldout.csv'
)

image_dev = pd.read_csv(
    image_dev_file
)

image_heldout = pd.read_csv(
    image_heldout_file
)

print(
    "Image development:",
    image_dev.shape
)

print(
    "Image held-out:",
    image_heldout.shape
)

Image development: (1513, 769)
Image held-out: (687, 769)


In [6]:
# Load structured features
structured_dev_file = (
    f'{processed_path}/'
    'structured_features_mlp.csv'
)

structured_heldout_file = (
    f'{processed_path}/'
    'structured_features_mlp_heldout.csv'
)

structured_dev = pd.read_csv(
    structured_dev_file
)

structured_heldout = pd.read_csv(
    structured_heldout_file
)

print(
    "Structured development:",
    structured_dev.shape
)

print(
    "Structured held-out:",
    structured_heldout.shape
)

Structured development: (1513, 129)
Structured held-out: (687, 129)


In [7]:
# Standardise study IDs
for df in [
    text_dev,
    text_heldout,
    image_dev,
    image_heldout,
    structured_dev,
    structured_heldout
]:

    df['study_id'] = (
        df['study_id']
        .astype(str)
    )

print(
    "Study IDs converted to strings."
)

Study IDs converted to strings.


In [8]:
# Check feature dimensions
text_features = [
    c for c in text_dev.columns
    if c != 'study_id'
]

image_features = [
    c for c in image_dev.columns
    if c != 'study_id'
]

structured_features = [
    c for c in structured_dev.columns
    if c != 'study_id'
]

print(
    "Text feature dimensions:",
    len(text_features)
)

print(
    "Image feature dimensions:",
    len(image_features)
)

print(
    "Structured feature dimensions:",
    len(structured_features)
)

Text feature dimensions: 768
Image feature dimensions: 768
Structured feature dimensions: 128


In [9]:
# Find common development studies
common_dev_ids = (
    set(text_dev['study_id'])
    &
    set(image_dev['study_id'])
    &
    set(structured_dev['study_id'])
)

print(
    "Common development studies:",
    len(common_dev_ids)
)

Common development studies: 1513


In [10]:
# Find common held-out studies
common_heldout_ids = (
    set(text_heldout['study_id'])
    &
    set(image_heldout['study_id'])
    &
    set(structured_heldout['study_id'])
)

print(
    "Common held-out studies:",
    len(common_heldout_ids)
)

Common held-out studies: 687


In [11]:
# Merge development modalities
fusion_dev = (
    text_dev[
        text_dev['study_id']
        .isin(common_dev_ids)
    ]
    .merge(
        image_dev[
            image_dev['study_id']
            .isin(common_dev_ids)
        ],
        on='study_id',
        how='inner',
        suffixes=(
            '_text',
            '_image'
        )
    )
    .merge(
        structured_dev[
            structured_dev['study_id']
            .isin(common_dev_ids)
        ],
        on='study_id',
        how='inner'
    )
)

print(
    "Fused development shape:",
    fusion_dev.shape
)

Fused development shape: (1513, 1665)


In [12]:
# Merge held-out modalities
fusion_heldout = (
    text_heldout[
        text_heldout['study_id']
        .isin(common_heldout_ids)
    ]
    .merge(
        image_heldout[
            image_heldout['study_id']
            .isin(common_heldout_ids)
        ],
        on='study_id',
        how='inner',
        suffixes=(
            '_text',
            '_image'
        )
    )
    .merge(
        structured_heldout[
            structured_heldout['study_id']
            .isin(common_heldout_ids)
        ],
        on='study_id',
        how='inner'
    )
)

print(
    "Fused held-out shape:",
    fusion_heldout.shape
)

Fused held-out shape: (687, 1665)


In [13]:
# Verify alignment
print(
    "Development unique studies:",
    fusion_dev['study_id'].nunique()
)

print(
    "Held-out unique studies:",
    fusion_heldout['study_id'].nunique()
)

print(
    "\nDuplicate development IDs:",
    fusion_dev['study_id']
    .duplicated()
    .sum()
)

print(
    "Duplicate held-out IDs:",
    fusion_heldout['study_id']
    .duplicated()
    .sum()
)

Development unique studies: 1513
Held-out unique studies: 687

Duplicate development IDs: 0
Duplicate held-out IDs: 0


In [14]:
# Load target labels
structured_data = pd.read_csv(
    f'{processed_path}/'
    'structured_processed.csv'
)

structured_data['study_id'] = (
    structured_data['study_id']
    .astype(str)
)

label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

available_labels = [
    label
    for label in label_columns
    if label in structured_data.columns
]

print(
    "Labels:",
    available_labels
)

Labels: ['No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [15]:
# Prepare labels
for label in available_labels:

    structured_data[label] = (
        pd.to_numeric(
            structured_data[label],
            errors='coerce'
        )
        .fillna(0)
    )

    structured_data[label] = (
        structured_data[label] == 1
    ).astype(np.float32)

In [16]:
# Add development labels
fusion_dev = fusion_dev.merge(
    structured_data[
        ['study_id']
        + available_labels
    ],
    on='study_id',
    how='left'
)

print(
    "Development with labels:",
    fusion_dev.shape
)

Development with labels: (1513, 1672)


In [17]:
# Add held-out labels
fusion_heldout = fusion_heldout.merge(
    structured_data[
        ['study_id']
        + available_labels
    ],
    on='study_id',
    how='left'
)

print(
    "Held-out with labels:",
    fusion_heldout.shape
)

Held-out with labels: (687, 1672)


In [18]:
# Build feature matrices
X_dev_text = (
    fusion_dev[
        text_features
    ]
    .values
    .astype(np.float32)
)

X_dev_image = (
    fusion_dev[
        image_features
    ]
    .values
    .astype(np.float32)
)

X_dev_structured = (
    fusion_dev[
        structured_features
    ]
    .values
    .astype(np.float32)
)

X_heldout_text = (
    fusion_heldout[
        text_features
    ]
    .values
    .astype(np.float32)
)

X_heldout_image = (
    fusion_heldout[
        image_features
    ]
    .values
    .astype(np.float32)
)

X_heldout_structured = (
    fusion_heldout[
        structured_features
    ]
    .values
    .astype(np.float32)
)

print(
    "Text:",
    X_dev_text.shape
)

print(
    "Image:",
    X_dev_image.shape
)

print(
    "Structured:",
    X_dev_structured.shape
)

Text: (1513, 768)
Image: (1513, 768)
Structured: (1513, 128)


In [19]:
# Concatenate modalities
X_dev = np.concatenate(
    [
        X_dev_text,
        X_dev_image,
        X_dev_structured
    ],
    axis=1
)

X_heldout = np.concatenate(
    [
        X_heldout_text,
        X_heldout_image,
        X_heldout_structured
    ],
    axis=1
)

print(
    "Final development feature matrix:",
    X_dev.shape
)

print(
    "Final held-out feature matrix:",
    X_heldout.shape
)

Final development feature matrix: (1513, 1664)
Final held-out feature matrix: (687, 1664)


In [20]:
# Create labels
y_dev = (
    fusion_dev[
        available_labels
    ]
    .values
    .astype(np.float32)
)

y_heldout = (
    fusion_heldout[
        available_labels
    ]
    .values
    .astype(np.float32)
)

print(
    "Development labels:",
    y_dev.shape
)

print(
    "Held-out labels:",
    y_heldout.shape
)

Development labels: (1513, 7)
Held-out labels: (687, 7)


In [21]:
# Check missing values
print(
    "Missing values in development:",
    np.isnan(X_dev).sum()
)

print(
    "Missing values in held-out:",
    np.isnan(X_heldout).sum()
)

Missing values in development: 0
Missing values in held-out: 0


In [22]:
# Replace any remaining invalid values
X_dev = np.nan_to_num(
    X_dev,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_heldout = np.nan_to_num(
    X_heldout,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

print(
    "Feature matrices cleaned."
)

Feature matrices cleaned.


In [23]:
# Scale fusion features
fusion_scaler = StandardScaler()

X_dev_scaled = (
    fusion_scaler
    .fit_transform(X_dev)
    .astype(np.float32)
)

X_heldout_scaled = (
    fusion_scaler
    .transform(X_heldout)
    .astype(np.float32)
)

print(
    "Scaled development:",
    X_dev_scaled.shape
)

print(
    "Scaled held-out:",
    X_heldout_scaled.shape
)

Scaled development: (1513, 1664)
Scaled held-out: (687, 1664)


In [24]:
# Internal validation split
from sklearn.model_selection import train_test_split

train_idx, validation_idx = (
    train_test_split(
        np.arange(
            len(X_dev_scaled)
        ),
        test_size=0.15,
        random_state=SEED
    )
)

X_train = (
    X_dev_scaled[
        train_idx
    ]
)

X_validation = (
    X_dev_scaled[
        validation_idx
    ]
)

y_train = (
    y_dev[
        train_idx
    ]
)

y_validation = (
    y_dev[
        validation_idx
    ]
)

print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_validation.shape
)

Training: (1286, 1664)
Validation: (227, 1664)


In [25]:
# Simple SMOTE-style balancing
label_signature = [
    '_'.join(
        str(int(value))
        for value in row
    )
    for row in y_train
]

print(
    "Unique label combinations:",
    len(set(label_signature))
)

Unique label combinations: 64


In [26]:
# Random oversampling
ros = RandomOverSampler(
    random_state=SEED
)

X_train_balanced, signature_balanced = (
    ros.fit_resample(
        X_train,
        label_signature
    )
)

print(
    "Original training size:",
    X_train.shape[0]
)

print(
    "Balanced training size:",
    X_train_balanced.shape[0]
)

Original training size: 1286
Balanced training size: 17856


In [27]:
# Recover multilabel targets
y_train_balanced = np.array([
    [
        float(value)
        for value in signature.split('_')
    ]
    for signature in signature_balanced
],
dtype=np.float32
)

print(
    "Balanced features:",
    X_train_balanced.shape
)

print(
    "Balanced labels:",
    y_train_balanced.shape
)

Balanced features: (17856, 1664)
Balanced labels: (17856, 7)


In [28]:
# Convert to tensors
X_train_tensor = torch.tensor(
    X_train_balanced,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_balanced,
    dtype=torch.float32
)

X_validation_tensor = torch.tensor(
    X_validation,
    dtype=torch.float32
)

y_validation_tensor = torch.tensor(
    y_validation,
    dtype=torch.float32
)

print(
    X_train_tensor.shape
)

print(
    y_train_tensor.shape
)

torch.Size([17856, 1664])
torch.Size([17856, 7])


In [29]:
# DataLoaders
BATCH_SIZE = 32

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

validation_dataset = TensorDataset(
    X_validation_tensor,
    y_validation_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

Training batches: 558
Validation batches: 8


In [30]:
# Define Layer 1 fusion model
class MultimodalFusionModel(
    nn.Module
):

    def __init__(
        self,
        input_size,
        num_labels
    ):

        super().__init__()

        self.fusion_layer = nn.Sequential(
            nn.Linear(
                input_size,
                512
            ),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(
                512,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(
            128,
            num_labels
        )

    def forward(
        self,
        x
    ):

        features = (
            self.fusion_layer(x)
        )

        logits = (
            self.classifier(features)
        )

        return logits, features


print(
    "Fusion model created."
)

Fusion model created.


In [31]:
# Create model
fusion_model = (
    MultimodalFusionModel(
        input_size=X_train.shape[1],
        num_labels=len(
            available_labels
        )
    )
    .to(device)
)

print(
    fusion_model
)

MultimodalFusionModel(
  (fusion_layer): Sequential(
    (0): Linear(in_features=1664, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
  )
  (classifier): Linear(in_features=128, out_features=7, bias=True)
)


In [32]:
# Class weights
positive_counts = (
    y_train_balanced.sum(axis=0)
)

negative_counts = (
    len(y_train_balanced)
    - positive_counts
)

pos_weights = (
    negative_counts
    /
    np.maximum(
        positive_counts,
        1
    )
)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

print(
    pd.DataFrame({
        'label': available_labels,
        'positive': positive_counts,
        'negative': negative_counts,
        'weight':
            pos_weights.cpu().numpy()
    })
)

              label  positive  negative     weight
0        No Finding     558.0   17298.0  31.000000
1   Support Devices    9207.0    8649.0   0.939394
2  Pleural Effusion    8928.0    8928.0   1.000000
3      Lung Opacity    8649.0    9207.0   1.064516
4       Atelectasis    8370.0    9486.0   1.133333
5      Cardiomegaly    8370.0    9486.0   1.133333
6             Edema    8370.0    9486.0   1.133333


In [33]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=0.001
)

print(
    "Fusion optimizer ready."
)

Fusion optimizer ready.


In [34]:
# Training function
def train_fusion_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            device
        )

        y_batch = y_batch.to(
            device
        )

        optimizer.zero_grad()

        logits, _ = model(
            X_batch
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * X_batch.size(0)
        )

    return (
        total_loss
        /
        len(loader.dataset)
    )

In [35]:
# Evaluation function
def evaluate_fusion(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0

    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device
            )

            y_batch = y_batch.to(
                device
            )

            logits, _ = model(
                X_batch
            )

            probabilities = (
                torch.sigmoid(logits)
            )

            loss = criterion(
                logits,
                y_batch
            )

            total_loss += (
                loss.item()
                * X_batch.size(0)
            )

            all_labels.append(
                y_batch
                .cpu()
                .numpy()
            )

            all_probabilities.append(
                probabilities
                .cpu()
                .numpy()
            )

    y_true = np.vstack(
        all_labels
    )

    y_prob = np.vstack(
        all_probabilities
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    return (
        total_loss /
        len(loader.dataset),
        y_true,
        y_prob,
        y_pred
    )

In [36]:
# Train Layer 1 fusion model
EPOCHS = 10

fusion_history = []

for epoch in range(EPOCHS):

    train_loss = (
        train_fusion_epoch(
            fusion_model,
            train_loader,
            optimizer,
            criterion,
            device
        )
    )

    (
        validation_loss,
        y_true,
        y_prob,
        y_pred
    ) = evaluate_fusion(
        fusion_model,
        validation_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: "
        f"{validation_loss:.4f}"
    )

    fusion_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'validation_loss':
            validation_loss
    })

Epoch 1/10 | Train loss: 0.2068 | Validation loss: 0.9037
Epoch 2/10 | Train loss: 0.0552 | Validation loss: 1.2581
Epoch 3/10 | Train loss: 0.0442 | Validation loss: 1.7177
Epoch 4/10 | Train loss: 0.0314 | Validation loss: 1.3380
Epoch 5/10 | Train loss: 0.0334 | Validation loss: 1.9616
Epoch 6/10 | Train loss: 0.0921 | Validation loss: 1.5851
Epoch 7/10 | Train loss: 0.0984 | Validation loss: 2.4809
Epoch 8/10 | Train loss: 0.2425 | Validation loss: 6.3856
Epoch 9/10 | Train loss: 0.2695 | Validation loss: 11.2494
Epoch 10/10 | Train loss: 0.1541 | Validation loss: 11.2211


In [37]:
# Training history
fusion_history_df = pd.DataFrame(
    fusion_history
)

display(
    fusion_history_df
)

,epoch,train_loss,validation_loss
0,1,0.206788,0.903700
1,2,0.055168,1.258084
2,3,0.044222,1.717746
3,4,0.031425,1.337957
4,5,0.033362,1.961638
5,6,0.092129,1.585068
6,7,0.098366,2.480865
7,8,0.242478,6.385639
8,9,0.269508,11.249403
9,10,0.154087,11.221069


In [38]:
# Evaluation
fusion_metrics = []

for i, label in enumerate(
    available_labels
):

    true_values = (
        y_true[:, i]
        .astype(int)
    )

    probabilities = (
        y_prob[:, i]
    )

    predicted_values = (
        probabilities >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        true_values,
        predicted_values
    )

    precision = precision_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    recall = recall_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    f1 = f1_score(
        true_values,
        predicted_values,
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            true_values,
            probabilities
        )

    except ValueError:

        roc_auc = np.nan

    fusion_metrics.append({
        'label': label,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    })

fusion_metrics_df = pd.DataFrame(
    fusion_metrics
)

display(
    fusion_metrics_df
)

,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.911894,0.819672,0.847458,0.833333,0.918382
1,Support Devices,0.766520,0.623656,0.763158,0.686391,0.869990
2,Pleural Effusion,0.872247,0.736842,0.750000,0.743363,0.938074
3,Lung Opacity,0.947137,0.950820,0.865672,0.906250,0.982183
4,Atelectasis,0.885463,0.796875,0.796875,0.796875,0.950633
5,Cardiomegaly,0.920705,0.930233,0.727273,0.816327,0.946934
6,Edema,0.969163,0.791667,0.904762,0.844444,0.988442


In [39]:
# Overall Layer 1 metrics
y_true_binary = (
    y_true == 1
).astype(int)

y_pred_binary = (
    y_pred == 1
).astype(int)

overall_fusion_metrics = pd.DataFrame({

    'metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC'
    ],

    'value': [

        accuracy_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten()
        ),

        precision_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        recall_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        f1_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            zero_division=0
        ),

        roc_auc_score(
            y_true_binary,
            y_prob,
            average='macro'
        )
    ]
})

display(
    overall_fusion_metrics
)

,metric,value
0,Accuracy,0.896161
1,Precision,0.789082
2,Recall,0.798995
3,F1,0.794007
4,ROC-AUC,0.942091


In [40]:
# Save fusion metrics
fusion_metrics_file = (
    f'{processed_path}/'
    'fusion_layer1_metrics.csv'
)

fusion_metrics_df.to_csv(
    fusion_metrics_file,
    index=False
)

print(
    "Saved:",
    fusion_metrics_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/fusion_layer1_metrics.csv


In [41]:
# Save training history
fusion_history_file = (
    f'{processed_path}/'
    'fusion_layer1_training_history.csv'
)

fusion_history_df.to_csv(
    fusion_history_file,
    index=False
)

print(
    "Saved:",
    fusion_history_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/fusion_layer1_training_history.csv


In [42]:
# Save scaler
import joblib

fusion_scaler_file = (
    f'{model_path}/'
    'fusion_scaler.pkl'
)

joblib.dump(
    fusion_scaler,
    fusion_scaler_file
)

print(
    "Saved:",
    fusion_scaler_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/models/fusion_scaler.pkl


In [43]:
# Save Layer 1 model
fusion_model_file = (
    f'{model_path}/'
    'multimodal_fusion_layer1.pt'
)

torch.save(
    {
        'model_state_dict':
            fusion_model.state_dict(),

        'input_size':
            X_train.shape[1],

        'num_labels':
            len(available_labels),

        'labels':
            available_labels,

        'text_dimension':
            len(text_features),

        'image_dimension':
            len(image_features),

        'structured_dimension':
            len(structured_features),

        'fusion_dimension':
            128
    },
    fusion_model_file
)

print(
    "Saved Layer 1 model:"
)

print(
    fusion_model_file
)

Saved Layer 1 model:
/content/drive/MyDrive/dissertation_project/data/models/multimodal_fusion_layer1.pt


In [44]:
# Extract Layer 1 fused representations
def extract_fusion_features(
    model,
    X,
    device,
    batch_size=32
):

    tensor_data = torch.tensor(
        X,
        dtype=torch.float32
    )

    dataset = TensorDataset(
        tensor_data
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    features = []

    with torch.no_grad():

        for batch in loader:

            X_batch = (
                batch[0]
                .to(device)
            )

            _, feature_batch = model(
                X_batch
            )

            features.append(
                feature_batch
                .cpu()
                .numpy()
            )

    return np.vstack(
        features
    )

In [45]:
# Extract development fusion features
development_fusion_features = (
    extract_fusion_features(
        fusion_model,
        X_dev_scaled,
        device
    )
)

print(
    "Development fusion features:",
    development_fusion_features.shape
)

Development fusion features: (1513, 128)


In [46]:
# Extract held-out fusion features
heldout_fusion_features = (
    extract_fusion_features(
        fusion_model,
        X_heldout_scaled,
        device
    )
)

print(
    "Held-out fusion features:",
    heldout_fusion_features.shape
)

Held-out fusion features: (687, 128)


In [47]:
# Save development fusion features
fusion_dev_features_df = pd.DataFrame(
    development_fusion_features,
    columns=[
        f'fusion_feature_{i}'
        for i in range(
            development_fusion_features.shape[1]
        )
    ]
)

fusion_dev_features_df.insert(
    0,
    'study_id',
    fusion_dev[
        'study_id'
    ].values
)

fusion_dev_features_file = (
    f'{processed_path}/'
    'fusion_features_layer1.csv'
)

fusion_dev_features_df.to_csv(
    fusion_dev_features_file,
    index=False
)

print(
    "Saved:",
    fusion_dev_features_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/fusion_features_layer1.csv


In [48]:
heldout_fusion_features_df = pd.DataFrame(
    heldout_fusion_features,
    columns=[f'fusion_feature_{i}' for i in range(heldout_fusion_features.shape[1])]
)
heldout_fusion_features_df.insert(0, 'study_id', fusion_heldout['study_id'].values)

heldout_fusion_features_file = f'{processed_path}/fusion_features_layer1_heldout.csv'
heldout_fusion_features_df.to_csv(heldout_fusion_features_file, index=False)

print("Saved:", heldout_fusion_features_file)
print("Shape:", heldout_fusion_features_df.shape)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/fusion_features_layer1_heldout.csv
Shape: (687, 129)
